# MVP4 – Classificação de Risco na Gestação

**Disciplina:** Engenharia de Sistemas de Software Inteligentes – PUC-Rio
**Autor:** André Araujo
**Repositório:** https://github.com/artisaj/mvp4-risco-gestacional

Este notebook é o relatório do processo de criação do modelo de machine learning do MVP. Ele cobre o problema, a carga e a preparação dos dados, a modelagem com quatro algoritmos clássicos, a otimização, a avaliação e a exportação do modelo, que depois é servido por uma aplicação full stack.

**Sumário**
1. Definição do problema
2. Configuração do ambiente
3. Carga dos dados
4. Análise exploratória
5. Limpeza dos dados
6. Separação entre treino e teste (holdout)
7. Validação cruzada
8. Modelagem – linha de base
9. Otimização de hiperparâmetros
10. Avaliação no conjunto de teste
11. Requisitos de desempenho do modelo
12. Exportação do modelo
13. Segurança e anonimização dos dados
14. Análise dos resultados
15. Conclusão

## 1. Definição do problema

A mortalidade materna ainda é um problema grave de saúde pública, sobretudo em regiões com pouco acesso a acompanhamento pré-natal. A redução dessas mortes é uma das metas do Objetivo de Desenvolvimento Sustentável 3 da ONU. Boa parte das complicações pode ser evitada quando as gestantes de maior risco são identificadas cedo e encaminhadas para acompanhamento especializado.

O dataset **Maternal Health Risk** ([UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/863/maternal+health+risk)) foi coletado em hospitais, clínicas comunitárias e postos de saúde materna de áreas rurais de Bangladesh, por meio de um sistema de monitoramento de risco baseado em IoT. Cada linha corresponde às medições de uma gestante e ao nível de risco atribuído a ela.

**Objetivo:** construir um modelo que, a partir de seis sinais clínicos simples de medir, classifique a gestação em **risco baixo, médio ou alto**. É um problema de **aprendizado supervisionado** de **classificação multiclasse**: aprendemos com exemplos já rotulados e prevemos uma entre três categorias.

**Premissas:**
- O modelo é uma ferramenta de **apoio à triagem** e não substitui a avaliação de um profissional de saúde.
- Em uma triagem de risco, deixar de identificar uma gestante de **alto risco** (falso negativo) é muito mais grave do que alertar sem necessidade. Por isso daremos atenção especial ao *recall* dessa classe.

**Atributos:**

| Atributo | Descrição | Unidade |
|---|---|---|
| `Age` | Idade da gestante | anos |
| `SystolicBP` | Pressão arterial sistólica (a "máxima") | mmHg |
| `DiastolicBP` | Pressão arterial diastólica (a "mínima") | mmHg |
| `BS` | Glicemia (açúcar no sangue) | mmol/L |
| `BodyTemp` | Temperatura corporal | °F |
| `HeartRate` | Frequência cardíaca em repouso | bpm |
| **`RiskLevel`** | **Classe alvo:** `low risk`, `mid risk`, `high risk` | – |

## 2. Configuração do ambiente

Importamos as bibliotecas e definimos as constantes usadas no notebook. Todas as configurações ficam concentradas aqui, e não espalhadas pelo código, o que facilita a manutenção.

- **pandas:** manipulação de tabelas (DataFrames).
- **matplotlib / seaborn:** gráficos.
- **scikit-learn:** algoritmos de machine learning, transformações, validação e métricas.
- **joblib:** salva o modelo treinado em arquivo.

A `SEED` (semente aleatória) fixa todas as operações que envolvem sorteio, como a divisão treino/teste e a validação cruzada. Assim o notebook produz os mesmos resultados a cada execução.

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [ ]:
SEED = 7
DATASET_URL = (
    "https://raw.githubusercontent.com/artisaj/mvp4-risco-gestacional/"
    "main/data/maternal_health_risk.csv"
)
FEATURES = ["Age", "SystolicBP", "DiastolicBP", "BS", "BodyTemp", "HeartRate"]
TARGET = "RiskLevel"
CLASS_ORDER = ["low risk", "mid risk", "high risk"]
HIGH_RISK = "high risk"

TEST_SIZE = 0.20
N_FOLDS = 10
N_REPEATS = 3
SCORING = "f1_macro"

OUTPUT_DIR = Path("artefatos")
MODEL_FILE = OUTPUT_DIR / "modelo_risco_gestacional.pkl"
TEST_DATA_FILE = OUTPUT_DIR / "dataset_teste.csv"

print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__}")

## 3. Carga dos dados

O dataset é lido diretamente do repositório do projeto no GitHub pela URL *raw*, ou seja, a do arquivo puro, sem a página do GitHub em volta. Assim o notebook roda em qualquer ambiente, sem precisar fazer upload de arquivos.

In [ ]:
df = pd.read_csv(DATASET_URL)
print(f"{df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

## 4. Análise exploratória

Antes de modelar, é preciso conhecer os dados: tipos, valores faltantes, distribuições, valores anômalos e o balanceamento das classes.

In [ ]:
df.info()

Todas as seis variáveis de entrada são numéricas e não há valores nulos. A variável alvo (`RiskLevel`) é texto. O scikit-learn aceita rótulos em texto na classificação, então não é preciso convertê-los em números.

In [ ]:
df.describe().T

Algumas observações sobre as estatísticas:

- **`HeartRate`** tem mínimo de **7 bpm**, valor incompatível com a vida e quase certamente um erro de digitação (provavelmente 70 ou 77).
- **`BodyTemp`** está em Fahrenheit, e a maioria dos registros vale exatamente 98 °F (≈ 36,7 °C, temperatura normal). Os poucos valores acima indicam febre.
- **`BS`** tem mediana 7,5 e máximo 19 mmol/L: há gestantes com glicemia bem elevada.

Em seguida, verificamos registros duplicados.

In [ ]:
n_duplicated = df.duplicated().sum()
print(f"Linhas duplicadas: {n_duplicated} de {len(df)} ({n_duplicated / len(df):.0%})")

**Mais da metade das linhas são cópias exatas de outras.** Isso é crítico. Se mantivermos as duplicatas, a mesma linha pode cair tanto no treino quanto no teste, e o modelo seria avaliado em exemplos que já viu durante o treino. Isso é **vazamento de dados** (*data leakage*) e produz métricas otimistas demais, que não se repetiriam com gestantes novas. As duplicatas serão removidas na etapa de limpeza.

Vejamos a distribuição das classes.

In [ ]:
def plot_class_distribution(data, title):
    ax = sns.countplot(data=data, x=TARGET, order=CLASS_ORDER, hue=TARGET,
                       palette="Set2", legend=False)
    ax.set(title=title, xlabel="Nível de risco", ylabel="Quantidade")
    for container in ax.containers:
        ax.bar_label(container)
    plt.show()


plot_class_distribution(df, "Distribuição das classes (dataset original)")

As três classes têm tamanhos da mesma ordem de grandeza. O desbalanceamento é leve, e não precisamos de técnicas de reamostragem. Vejamos como cada atributo se distribui em cada nível de risco.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.flat, FEATURES):
    sns.boxplot(data=df, x=TARGET, y=feature, order=CLASS_ORDER, hue=TARGET,
                palette="Set2", legend=False, ax=ax)
    ax.set(title=feature, xlabel="")
fig.suptitle("Atributos por nível de risco", fontsize=14)
plt.tight_layout()
plt.show()

Os boxplots mostram que:

- **`BS` (glicemia)** é o atributo que mais separa o alto risco: quase todos os valores elevados estão nessa classe.
- **Pressão sistólica e diastólica** também são bem mais altas no grupo de alto risco.
- **Risco baixo e médio se sobrepõem bastante** em quase todos os atributos. Já dá para prever que distinguir essas duas classes será a parte mais difícil.

O mapa de calor abaixo mostra a correlação entre os atributos numéricos.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1)
plt.title("Correlação entre os atributos")
plt.show()

As pressões sistólica e diastólica são fortemente correlacionadas (≈ 0,79), o que é esperado fisiologicamente. As demais correlações são fracas ou moderadas. Não há redundância que justifique remover atributos, e com apenas seis variáveis manteremos todas.

## 5. Limpeza dos dados

Com base na análise exploratória, fazemos duas limpezas:

1. **Remoção das linhas duplicadas**, para evitar vazamento entre treino e teste.
2. **Remoção dos registros com frequência cardíaca abaixo de 40 bpm**, valor fisiologicamente impossível para uma gestante em repouso e, portanto, erro de medição ou de digitação.

In [ ]:
MIN_PLAUSIBLE_HEART_RATE = 40

df_clean = (
    df.drop_duplicates()
      .query("HeartRate >= @MIN_PLAUSIBLE_HEART_RATE")
      .reset_index(drop=True)
)
print(f"Antes: {len(df)} linhas | Depois: {len(df_clean)} linhas")
plot_class_distribution(df_clean, "Distribuição das classes (após limpeza)")

Restam 451 registros únicos. Após a limpeza, a classe `low risk` passou a ter cerca do dobro de exemplos de cada uma das outras. Ainda é um desbalanceamento moderado, que trataremos de duas formas: com divisões **estratificadas**, que mantêm a proporção das classes em cada parte, e com uma métrica que dá o mesmo peso a todas as classes (F1-macro, explicada adiante).

Por fim, verificamos se há gestantes com **exatamente os mesmos atributos e rótulos diferentes**.

In [ ]:
n_conflicting = df_clean.duplicated(subset=FEATURES, keep=False).sum()
print(f"Registros com atributos idênticos a outro, mas com rótulo diferente: {n_conflicting}")

Há registros com as mesmas medições e níveis de risco diferentes. Isso indica que **os seis atributos não bastam para determinar o risco com certeza**: o profissional que fez a rotulação provavelmente considerou outras informações, como histórico clínico e semana gestacional. Mantemos esses registros, porque eles refletem a realidade, mas eles impõem um **teto ao desempenho** de qualquer modelo neste dataset.

## 6. Separação entre treino e teste (holdout)

Separamos os dados em duas partes:

- **Treino (80%):** usado para escolher e treinar o modelo.
- **Teste (20%):** guardado e usado **uma única vez**, no final, para estimar como o modelo se sairia com gestantes novas.

O parâmetro `stratify=y` garante que as três classes apareçam no teste na mesma proporção do conjunto completo.

In [ ]:
X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)
print(f"Treino: {len(X_train)} registros | Teste: {len(X_test)} registros")

## 7. Validação cruzada

Para comparar algoritmos e escolher hiperparâmetros, **não** usamos o conjunto de teste. Se usássemos, ele deixaria de ser uma avaliação imparcial. Em vez disso, aplicamos a **validação cruzada (*cross-validation*)** dentro do treino:

1. O treino é dividido em 10 partes (*folds*).
2. O modelo é treinado em 9 partes e avaliado na parte restante.
3. Isso se repete 10 vezes, cada vez avaliando numa parte diferente.

O resultado é a média (e o desvio padrão) das 10 avaliações. Como o dataset é pequeno, os resultados variam conforme o sorteio das partes, então **repetimos o processo 3 vezes** com sorteios diferentes (`RepeatedStratifiedKFold`), num total de 30 avaliações por modelo. Isso deixa a comparação mais estável.

**Métrica de comparação – F1-macro:** o F1 combina *precision* (dos casos que o modelo apontou como de uma classe, quantos eram de fato) e *recall* (dos casos de uma classe, quantos o modelo encontrou). A versão *macro* calcula o F1 de cada classe e tira a média simples. Assim, as classes `mid risk` e `high risk`, que são menores, pesam tanto quanto `low risk`. A acurácia sozinha poderia esconder um modelo que acerta muito a classe majoritária e erra as demais.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=SEED)

## 8. Modelagem – linha de base

### Os algoritmos

| Algoritmo | Como funciona, em resumo |
|---|---|
| **KNN** (*K-Nearest Neighbors*) | Classifica uma gestante nova olhando as *K* gestantes mais parecidas do treino (as mais "próximas" em distância) e adotando a classe mais frequente entre elas. |
| **Árvore de Classificação** (CART) | Aprende uma sequência de perguntas do tipo "BS > 7,9?" que divide os dados em grupos cada vez mais homogêneos. É fácil de interpretar. |
| **Naive Bayes** (Gaussiano) | Usa probabilidade (teorema de Bayes) supondo que cada atributo segue uma distribuição normal e que os atributos são independentes entre si. É simples e rápido. |
| **SVM** (*Support Vector Machine*) | Procura as fronteiras que separam as classes com a maior margem possível. Com o *kernel* RBF, essas fronteiras podem ser curvas. |

### As transformações

Os atributos estão em escalas bem diferentes: a pressão vai de 70 a 160, e a glicemia de 6 a 19. Algoritmos baseados em **distância** (KNN e SVM) dariam peso indevido aos atributos de números maiores. Testamos três variações para cada algoritmo:

- **Original:** sem transformação.
- **Padronização** (`StandardScaler`): cada atributo passa a ter média 0 e desvio padrão 1.
- **Normalização** (`MinMaxScaler`): cada atributo é colocado no intervalo [0, 1].

### Por que pipelines?

Um **pipeline** encadeia a transformação e o modelo num único objeto. Em cada rodada da validação cruzada, o *scaler* aprende média, desvio, mínimo e máximo **apenas com os folds de treino** e só então é aplicado ao fold de validação. Sem o pipeline, seria fácil calcular essas estatísticas com todos os dados e vazar informação da validação para o treino. Além disso, o pipeline exportado no final já carrega a transformação junto: a aplicação só precisa enviar os valores brutos.

In [ ]:
def build_algorithms():
    # Algoritmos com hiperparâmetros padrão.
    return {
        "KNN": KNeighborsClassifier(),
        "CART": DecisionTreeClassifier(random_state=SEED),
        "NB": GaussianNB(),
        "SVM": SVC(random_state=SEED),
    }


def build_scalers():
    # Variações de transformação de dados. 'passthrough' significa nenhuma.
    return {
        "orig": "passthrough",
        "padr": StandardScaler(),
        "norm": MinMaxScaler(),
    }


def build_pipeline(scaler, model):
    return Pipeline([("scaler", scaler), ("model", model)])

In [ ]:
baseline_scores = {}
for algo_name, model in build_algorithms().items():
    for scaler_name, scaler in build_scalers().items():
        name = f"{algo_name}-{scaler_name}"
        pipeline = build_pipeline(scaler, model)
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
        baseline_scores[name] = scores
        print(f"{name:10s} F1-macro: {scores.mean():.3f} (± {scores.std():.3f})")

In [ ]:
plt.figure(figsize=(14, 5))
sns.boxplot(data=pd.DataFrame(baseline_scores), palette="Set2")
plt.title("Linha de base – F1-macro na validação cruzada (30 avaliações por modelo)")
plt.ylabel("F1-macro")
plt.xticks(rotation=45)
plt.show()

**Leitura da linha de base:**

- **SVM sem transformação** tem o pior desempenho de todos e melhora bastante com padronização. Isso confirma que o SVM é sensível à escala dos atributos.
- **KNN** também melhora com padronização ou normalização, pelo mesmo motivo: ele usa distâncias.
- **Árvore de Classificação e Naive Bayes** praticamente não mudam com as transformações. A árvore compara um atributo por vez com um limiar ("BS > 7,9?"), e mudar a escala não altera a ordem dos valores. O Naive Bayes Gaussiano estima média e variância de cada atributo, e essas estatísticas se ajustam junto com a escala.
- A dispersão (altura das caixas) é grande para todos os modelos: com poucos dados, o desempenho varia bastante conforme a amostra. Isso justifica o uso da validação cruzada repetida.

Até aqui todos os modelos usam hiperparâmetros padrão. O próximo passo é otimizá-los.

## 9. Otimização de hiperparâmetros

**Hiperparâmetros** são configurações que definimos antes do treino, como o número de vizinhos do KNN ou a profundidade máxima da árvore. Usamos o **`GridSearchCV`**, que testa todas as combinações de uma grade de valores e avalia cada uma com a mesma validação cruzada repetida. A **transformação dos dados também entra na grade**, então a busca escolhe ao mesmo tempo a melhor transformação e os melhores hiperparâmetros.

| Algoritmo | Hiperparâmetros testados |
|---|---|
| KNN | `n_neighbors`: quantos vizinhos consultar · `metric`: forma de medir a distância · `weights`: se vizinhos mais próximos pesam mais |
| CART | `max_depth`: profundidade máxima (limita a complexidade) · `min_samples_leaf`: mínimo de exemplos em cada folha · `criterion`: medida de "pureza" usada nas divisões |
| NB | `var_smoothing`: suavização que dá estabilidade às variâncias estimadas |
| SVM | `C`: rigidez da margem (quanto penalizar erros no treino) · `gamma`: alcance da influência de cada exemplo · `kernel`: formato da fronteira (linear ou RBF) |

**Exceção no SVM:** a linha de base mostrou que o SVM sem transformação tem desempenho muito inferior. Além disso, com dados em escalas tão diferentes e `C` alto, o kernel linear demora muito para convergir, e a busca levaria dezenas de minutos. Por isso, para o SVM testamos só dados padronizados ou normalizados. O `gamma` também só é variado no kernel RBF, porque o kernel linear não o utiliza, e testá-lo ali só repetiria combinações equivalentes.

Limitar a complexidade, com `max_depth` e `min_samples_leaf` na árvore, `n_neighbors` maior no KNN e `C` menor no SVM, ajuda a evitar o **overfitting**: o modelo que "decora" o treino e erra com dados novos.

In [ ]:
SCALER_OPTIONS = list(build_scalers().values())
SCALED_OPTIONS = [StandardScaler(), MinMaxScaler()]

PARAM_GRIDS = {
    "KNN": {
        "model__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
        "model__metric": ["euclidean", "manhattan"],
        "model__weights": ["uniform", "distance"],
    },
    "CART": {
        "model__max_depth": [None, 3, 5, 7, 10],
        "model__min_samples_leaf": [1, 2, 5, 10],
        "model__criterion": ["gini", "entropy"],
    },
    "NB": {
        "model__var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3],
    },
    # SVM: só com dados transformados, e gamma só se aplica ao kernel RBF.
    "SVM": [
        {
            "scaler": SCALED_OPTIONS,
            "model__kernel": ["rbf"],
            "model__C": [0.1, 1, 10, 100],
            "model__gamma": ["scale", 0.01, 0.1, 1],
        },
        {
            "scaler": SCALED_OPTIONS,
            "model__kernel": ["linear"],
            "model__C": [0.1, 1, 10, 100],
        },
    ],
}

In [ ]:
def tune_model(model, param_grid):
    # Acrescenta as opções de transformação a cada grade, exceto quando ela já define as suas.
    sub_grids = param_grid if isinstance(param_grid, list) else [param_grid]
    grid = [{"scaler": SCALER_OPTIONS, **sub_grid} for sub_grid in sub_grids]
    search = GridSearchCV(build_pipeline("passthrough", model), grid,
                          cv=cv, scoring=SCORING, n_jobs=-1)
    return search.fit(X_train, y_train)


searches = {}
for algo_name, model in build_algorithms().items():
    searches[algo_name] = tune_model(model, PARAM_GRIDS[algo_name])
    search = searches[algo_name]
    print(f"{algo_name:5s} F1-macro: {search.best_score_:.3f} | {search.best_params_}")

In [ ]:
best_algo = max(searches, key=lambda name: searches[name].best_score_)
best_model = searches[best_algo].best_estimator_

print(f"Melhor algoritmo na validação cruzada: {best_algo}")
best_model

O **modelo final é escolhido pelo resultado da validação cruzada**, e não pelo conjunto de teste. Assim o teste continua sendo uma estimativa imparcial.

A **Árvore de Classificação** obteve o melhor F1-macro (≈ 0,61), com profundidade máxima 5 e pelo menos 2 exemplos por folha. Ela usa os dados originais, sem transformação, o que é coerente com a observação da linha de base de que árvores não dependem da escala. Uma árvore com profundidade limitada tem ainda a vantagem de ser **interpretável**: é possível explicar a um profissional de saúde por que o modelo chegou a uma classificação.

O `GridSearchCV` já retreina automaticamente a melhor configuração com **todo o conjunto de treino** (`refit=True`), e é esse modelo que avaliamos a seguir.

## 10. Avaliação no conjunto de teste

Agora usamos o conjunto de teste, separado no início e nunca visto pelos modelos. Mostramos os quatro algoritmos otimizados apenas para comparação, já que a escolha foi feita na etapa anterior.

In [ ]:
def evaluate(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    return {
        "acuracia": accuracy_score(y_eval, y_pred),
        "f1_macro": f1_score(y_eval, y_pred, average="macro"),
        "recall_alto_risco": recall_score(y_eval, y_pred, labels=[HIGH_RISK], average="macro"),
    }


test_results = pd.DataFrame(
    {name: evaluate(search.best_estimator_, X_test, y_test) for name, search in searches.items()}
).T
test_results.round(3)

In [ ]:
y_pred = best_model.predict(X_test)
print(f"Modelo escolhido: {best_algo}\n")
print(classification_report(y_test, y_pred, labels=CLASS_ORDER, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=CLASS_ORDER, cmap="Blues")
plt.title(f"Matriz de confusão – {best_algo} (conjunto de teste)")
plt.grid(False)
plt.show()

**Como ler a matriz de confusão:** cada linha é a classe real e cada coluna a classe prevista. A diagonal mostra os acertos, e as demais células mostram qual classe foi confundida com qual.

- **Alto risco:** o modelo encontra 19 das 23 gestantes de alto risco (recall ≈ 0,83), que era o objetivo principal.
- **Risco baixo:** também vai bem (recall ≈ 0,89).
- **Risco médio é o ponto fraco:** só 7 das 21 gestantes foram classificadas corretamente (recall ≈ 0,33). A maioria dos erros foi para risco baixo, como a análise exploratória já sugeria, já que esses grupos têm valores muito parecidos. Do ponto de vista da triagem, é um erro menos grave do que deixar passar um alto risco, mas é uma limitação clara do modelo.

Por fim, vemos quais atributos a árvore considerou mais importantes.

In [ ]:
final_estimator = best_model.named_steps["model"]
if hasattr(final_estimator, "feature_importances_"):
    importances = pd.Series(final_estimator.feature_importances_, index=FEATURES).sort_values()
    importances.plot.barh(color="steelblue", figsize=(7, 4))
    plt.title(f"Importância dos atributos – {best_algo}")
    plt.xlabel("Importância")
    plt.show()
else:
    print(f"O algoritmo {best_algo} não fornece importância de atributos.")

A **glicemia (`BS`)** é de longe o atributo mais importante, responsável por quase metade das decisões da árvore. Em seguida vêm a **pressão sistólica** e a **temperatura corporal** (febre). Isso bate com a análise exploratória e com o conhecimento clínico: diabetes gestacional, hipertensão (pré-eclâmpsia) e infecções estão entre os principais fatores de risco na gestação. Como a pressão diastólica é muito correlacionada com a sistólica, a árvore praticamente só usa uma das duas.

## 11. Requisitos de desempenho do modelo

Para a aplicação, definimos **requisitos mínimos de desempenho** que qualquer modelo precisa atingir no conjunto de teste para ser colocado em produção. Os mesmos limites são verificados pelo **teste automatizado (PyTest)** do back-end. Se o modelo for substituído e o novo não atingir os limites, o teste falha e a implantação deve ser barrada.

| Métrica | Limite mínimo | Justificativa |
|---|---|---|
| Recall de alto risco | **0,80** | Requisito principal: no máximo 1 em cada 5 gestantes de alto risco pode passar despercebida. |
| Acurácia | **0,70** | Acerto geral mínimo para a triagem ser útil. |
| F1-macro | **0,60** | Garante desempenho razoável também nas classes de risco baixo e médio. |

Os limites ficam um pouco abaixo dos resultados obtidos, como margem de segurança, mas acima do que modelos mais fracos alcançam. O KNN e o Naive Bayes otimizados, por exemplo, não atingem o requisito de recall de alto risco.

In [ ]:
PERFORMANCE_REQUIREMENTS = {
    "acuracia": 0.70,
    "f1_macro": 0.60,
    "recall_alto_risco": 0.80,
}

final_metrics = evaluate(best_model, X_test, y_test)
for metric, threshold in PERFORMANCE_REQUIREMENTS.items():
    status = "OK" if final_metrics[metric] >= threshold else "REPROVADO"
    print(f"{metric:18s} {final_metrics[metric]:.3f} (mínimo {threshold:.2f}) -> {status}")

## 12. Exportação do modelo

Salvamos dois arquivos, que serão copiados para o back-end da aplicação:

1. **`modelo_risco_gestacional.pkl`:** o pipeline completo (transformação + árvore), serializado com `joblib`. A aplicação carrega esse arquivo e chama `predict` com os valores brutos digitados pelo usuário.
2. **`dataset_teste.csv`:** o conjunto de teste (atributos + classe real), usado pelo teste automatizado PyTest para verificar os requisitos de desempenho.

**Por que exportar o modelo treinado só com o treino, e não com todos os dados?** Se o retreinássemos com os 100% dos dados, o conjunto de teste estaria incluído no treino, e o teste automatizado mediria o desempenho em dados já vistos, deixando de proteger contra modelos ruins. Abrimos mão de 20% dos dados de treino em troca de um controle de qualidade confiável.

**Atenção:** um modelo do scikit-learn só pode ser carregado com a **mesma versão** da biblioteca usada para salvá-lo. Por isso a versão é exibida abaixo e fixada no `requirements.txt` do back-end.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(best_model, MODEL_FILE)
X_test.assign(**{TARGET: y_test}).to_csv(TEST_DATA_FILE, index=False)

print(f"Modelo salvo em:       {MODEL_FILE}")
print(f"Dataset de teste em:   {TEST_DATA_FILE}")
print(f"Versão do scikit-learn: {sklearn.__version__}")

Conferimos que o arquivo salvo funciona: carregamos o modelo do disco e fazemos uma predição para uma gestante hipotética, do mesmo jeito que o back-end fará.

In [ ]:
loaded_model = joblib.load(MODEL_FILE)

new_patient = pd.DataFrame([{
    "Age": 35, "SystolicBP": 140, "DiastolicBP": 90,
    "BS": 13.0, "BodyTemp": 98.0, "HeartRate": 70,
}])
print("Predição para a nova gestante:", loaded_model.predict(new_patient)[0])

No Google Colab, a célula abaixo baixa os dois arquivos para o seu computador. Fora do Colab, ela apenas avisa onde os arquivos estão.

In [ ]:
try:
    from google.colab import files

    files.download(str(MODEL_FILE))
    files.download(str(TEST_DATA_FILE))
except ImportError:
    print(f"Fora do Colab: arquivos disponíveis na pasta '{OUTPUT_DIR}'.")

## 13. Segurança e anonimização dos dados

Este problema envolve **dados de saúde**, que a Lei Geral de Proteção de Dados (LGPD, Lei 13.709/2018, art. 5º, II) classifica como **dados pessoais sensíveis**. Esses dados exigem base legal específica (art. 11) e um cuidado maior com segurança. Se o sistema fosse usado de verdade, as práticas abaixo deveriam ser aplicadas.

### No dataset (treino do modelo)

- **Minimização:** coletar apenas o necessário. O modelo usa seis medições e **não precisa** de nome, CPF, endereço ou prontuário. O dataset público já segue esse princípio.
- **Supressão de identificadores diretos:** se os dados viessem de um sistema hospitalar real, colunas como nome, CPF, telefone e número do prontuário seriam removidas antes de chegar à equipe de ciência de dados.
- **Pseudonimização:** quando for preciso rastrear um registro, por exemplo para corrigir um dado, o identificador real é substituído por um código (como um *hash* com "sal" secreto), e a tabela de correspondência fica guardada separadamente, com acesso restrito.
- **Generalização e k-anonimato:** mesmo sem nome, uma combinação rara de atributos (por exemplo, uma gestante de 70 anos de uma clínica pequena) pode permitir a reidentificação. Atributos como a idade podem ser **generalizados em faixas** (15–19, 20–24, …), de forma que cada combinação de atributos apareça em pelo menos *k* registros. Há um custo: menos detalhe nos dados pode significar um modelo um pouco menos preciso.
- **Perturbação / privacidade diferencial:** adicionar um ruído controlado aos dados ou às estatísticas publicadas, dificultando inferências sobre indivíduos.
- **Controle de acesso e criptografia:** dados de treino armazenados criptografados, com acesso restrito e registrado (logs de auditoria).

### Na aplicação (uso do modelo)

- **Não armazenar** os dados digitados para predição, ou, se for necessário (ex.: histórico), armazená-los sem identificação da paciente.
- **HTTPS** na comunicação entre front-end e back-end, para que as medições não trafeguem em texto aberto.
- **Validação das entradas** no back-end (tipos e faixas plausíveis), rejeitando dados malformados ou maliciosos.
- **Logs sem dados sensíveis:** registrar erros e uso, mas nunca as medições de uma paciente identificada.
- **Proteção do próprio modelo:** um modelo pode "vazar" informações dos dados de treino, por meio de ataques de inferência de pertencimento. Por isso o arquivo `.pkl` não deve ser exposto publicamente num sistema real, e o acesso à API deveria exigir autenticação.

Neste MVP, o dataset é público e já anonimizado na origem, então nenhuma dessas técnicas foi necessária no treino. Na aplicação, a validação de entradas foi implementada e os dados de predição não são armazenados.

## 14. Análise dos resultados

**Principais achados:**

- **Qualidade dos dados foi o ponto mais importante do projeto.** 55% das linhas do dataset original eram duplicadas. Sem a remoção, as métricas pareceriam bem melhores, mas por vazamento de dados entre treino e teste, e não por qualidade real do modelo.
- **A Árvore de Classificação foi o melhor algoritmo** na validação cruzada (F1-macro ≈ 0,61), seguida por SVM e KNN. O Naive Bayes ficou em último, provavelmente porque a premissa de independência entre os atributos não se sustenta: as pressões sistólica e diastólica são fortemente correlacionadas.
- **As transformações de escala só fizeram diferença para KNN e SVM**, os algoritmos baseados em distância, como a teoria prevê.
- **No conjunto de teste**, o modelo final atingiu acurácia ≈ 0,75, F1-macro ≈ 0,68 e **recall de alto risco ≈ 0,83**, atendendo aos três requisitos de desempenho definidos.
- **Glicemia, pressão sistólica e temperatura** são os atributos mais relevantes para a decisão, em linha com o conhecimento médico.

**Pontos de atenção:**

- **Poucos dados:** após a limpeza restaram 451 registros, e o teste tem só 91. Uma ou duas predições a mais ou a menos mudam as métricas em alguns pontos percentuais. Os resultados devem ser lidos como estimativas aproximadas.
- **Risco médio mal identificado:** o recall dessa classe no teste foi de apenas ≈ 0,33, e a maioria dos erros foi para risco baixo. Os seis atributos sozinhos não separam bem essas classes, e existem registros idênticos com rótulos diferentes, o que limita qualquer modelo. Na prática, um resultado "risco baixo" do modelo não deve dispensar o acompanhamento pré-natal de rotina.
- **Representatividade:** os dados vêm de uma região específica (áreas rurais de Bangladesh). Antes de aplicar o modelo em outra população, como a brasileira, seria necessário validá-lo com dados locais.
- **Uso responsável:** o modelo é um apoio à triagem, e a decisão clínica deve ser sempre de um profissional de saúde.

**Possíveis melhorias:** coletar mais dados e novos atributos (semana gestacional, histórico de gestações, IMC), testar modelos de *ensemble* (como Random Forest) e ajustar o limiar de decisão para favorecer ainda mais o recall do alto risco.

## 15. Conclusão

Este notebook percorreu todas as etapas de um projeto de classificação: definição do problema, carga dos dados por URL, análise exploratória, limpeza, separação holdout estratificada, comparação de quatro algoritmos clássicos (KNN, Árvore de Classificação, Naive Bayes e SVM) com três variações de transformação de dados em pipelines, otimização de hiperparâmetros com validação cruzada repetida, avaliação no conjunto de teste e exportação do modelo.

O modelo escolhido, uma **Árvore de Classificação** com profundidade máxima 5, identifica corretamente cerca de **83% das gestações de alto risco**, além de ser simples e interpretável, características valiosas num contexto de saúde. Os requisitos de desempenho definidos aqui serão verificados automaticamente pelo teste PyTest da aplicação, protegendo o sistema contra a implantação de um modelo pior no futuro.

O modelo exportado é carregado pelo back-end da aplicação full stack, que recebe as medições digitadas no front-end e exibe o nível de risco previsto.